# 07 — LightGBM Gradient Boosting & SHAP Explainability
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook trains, optimizes, and interprets the high-capacity **LightGBM** model:
1. **Large-Scale Tabular Learning:** Leverages histogram-based binning and leaf-wise (best-first) tree growth to efficiently handle millions of rows.
2. **Feature Store Ingestion:** Ingests the engineered feature store with lag, rolling, cyclical, promotional, and calendar signals.
3. **Chronological 80/20 Split:** Evaluates generalization strictly on unseen future time intervals.
4. **Model Training & Regularization:** Uses early stopping and leaf constraints to mitigate overfitting.
5. **Feature Importance (Gain):** Analyzes feature contribution to overall error reduction.
6. **Explainable AI (XAI) via SHAP:** Computes Shapley Additive exPlanations (`shap.TreeExplainer`) on a representative sample to extract global drivers and local anomaly breakdowns.
7. **Model Persistence:** Serializes both Python pickle (`models/lgbm_model.pkl`) and native portable text (`models/lgbm_model.txt`).


In [ ]:
import os
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# --- 1. Universal Path Resolution ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Prefer processed feature store, fallback to data/
fs_path = PROCESSED_DATA_DIR / 'feature_store.csv'
if not fs_path.exists():
    fs_path = DATA_DIR / 'feature_store.csv'

print(f"Loading feature store from: {fs_path}...")
df = pd.read_csv(fs_path, parse_dates=['date'])
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns.")


### 2. Feature Specification & Data Preparation
Define the exact feature set expected by the production forecasting pipeline.


In [ ]:
target = 'sales'

desired_features = [
    'day_of_week', 'month', 'is_weekend', 'is_month_start', 'is_month_end',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28',
    'expanding_mean_sales', 'dcoilwtico',
    'promo_lag_1', 'promo_lag_7', 'promo_rolling_mean_7',
    'is_holiday'
]

features = [f for f in desired_features if f in df.columns]
print(f"Using {len(features)} verified features for LightGBM training:")
print(features)

# Drop any NaN rows resulting from initial lag offsets
clean_df = df.dropna(subset=features + [target]).sort_values('date').reset_index(drop=True)
print(f"Clean rows available for modeling: {len(clean_df):,}")


### 3. Chronological Train/Test Partition (80/20)
Split time chronologically to avoid lookahead bias.


In [ ]:
train_size = int(len(clean_df) * 0.8)
train_df = clean_df.iloc[:train_size]
test_df = clean_df.iloc[train_size:]

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"X_train shape: {X_train.shape} | Dates: {train_df['date'].min().date()} to {train_df['date'].max().date()}")
print(f"X_test shape:  {X_test.shape}  | Dates: {test_df['date'].min().date()} to {test_df['date'].max().date()}")


### 4. Training LightGBM Booster
We train using leaf-wise tree growth with early stopping on the validation set.


In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 20,
    'verbosity': -1,
    'seed': 42
}

print("Training LightGBM model...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[test_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

preds = model.predict(X_test, num_iteration=model.best_iteration)


### 5. Row-Level Evaluation Metrics
Evaluate performance at the granular row level ($Store \times Product \times Day$).


In [ ]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mask = y_test > 0
mape = np.mean(np.abs((y_test[mask] - preds[mask]) / y_test[mask])) * 100
smape = np.mean(2.0 * np.abs(y_test - preds) / (np.abs(y_test) + np.abs(preds) + 1e-8)) * 100

print(f"=== LightGBM Granular Row-Level Metrics ===")
print(f"MAE:   {mae:,.2f}")
print(f"RMSE:  {rmse:,.2f}")
print(f"MAPE:  {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")


### 6. Feature Importance (Gain)
Plot feature importance measured by total information gain.


In [ ]:
plt.figure(figsize=(10, 8))
lgb.plot_importance(model, importance_type='gain', max_num_features=15, title='LightGBM Feature Importance (Gain)')
plt.tight_layout()

feat_imp_path = REPORTS_DIR / 'lgbm_feature_importance.png'
plt.savefig(feat_imp_path, dpi=300)
plt.show()
print(f"Saved feature importance plot to: {feat_imp_path}")


### 7. Explainable AI via SHAP (Shapley Additive exPlanations)
We compute Shapley values on a 500-sample test slice using `shap.TreeExplainer`.


In [ ]:
print("Sampling 500 test rows for SHAP explanation...")
X_sample = X_test.sample(n=500, random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Beeswarm summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP Beeswarm: Feature Impact on Sales Predictions", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Mean absolute SHAP values table
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': features, 'Mean_Abs_SHAP': mean_abs_shap})
shap_df = shap_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
print("\nTop 8 Features by Mean Absolute SHAP Value:")
print(shap_df.head(8).to_string(index=False))


### 8. Model Serialization (Pickle & Native Formats)
We save the trained booster in both Python pickle format (`models/lgbm_model.pkl`) and LightGBM's safe, portable text format (`models/lgbm_model.txt`).


In [ ]:
# 1. Pickle format
pkl_path = MODELS_DIR / 'lgbm_model.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(model, f)
print(f"Saved pickle model: {pkl_path}")

# 2. Native text format (secure, language-agnostic)
txt_path = MODELS_DIR / 'lgbm_model.txt'
model.save_model(str(txt_path))
print(f"Saved native text model: {txt_path}")

print("\nLightGBM training & explainability completed! Proceed to 08_model_comparison_and_evaluation.ipynb.")
